In [1]:

!pip install duckdb openpyxl

In [2]:

from google.colab import files

uploaded = files.upload()

Saving online_retail_II.csv to online_retail_II.csv


In [3]:
import duckdb

con = duckdb.connect("day5_retail_analysis.duckdb")

con.execute("""
    CREATE TABLE retail AS
    SELECT *
    FROM read_csv_auto('online_retail_II.csv')
""")

print("Dataset loaded successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset loaded successfully!


In [4]:
# Display the column names and data types
print(con.execute("DESCRIBE retail").df())

# Display the first 5 rows
print(con.execute("SELECT * FROM retail LIMIT 5").df())

   column_name column_type null   key default extra
0      Invoice     VARCHAR  YES  None    None  None
1    StockCode     VARCHAR  YES  None    None  None
2  Description     VARCHAR  YES  None    None  None
3     Quantity      BIGINT  YES  None    None  None
4  InvoiceDate   TIMESTAMP  YES  None    None  None
5        Price      DOUBLE  YES  None    None  None
6  Customer ID      DOUBLE  YES  None    None  None
7      Country     VARCHAR  YES  None    None  None
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United K

In [6]:

monthly_summary = con.execute("""
    SELECT
        DATE_TRUNC('month', InvoiceDate) AS month,
        Country,
        SUM(Quantity * Price) AS revenue,
        COUNT(DISTINCT Invoice) AS orders,
        SUM(Quantity * Price)
            / NULLIF(COUNT(DISTINCT Invoice), 0) AS average_order_value
    FROM retail
    WHERE Quantity > 0
      AND Price > 0
      AND Invoice NOT LIKE 'C%'
    GROUP BY
        DATE_TRUNC('month', InvoiceDate),
        Country
    ORDER BY
        month,
        Country
""").df()

print(monthly_summary.head(10))


       month          Country   revenue  orders  average_order_value
0 2009-12-01        Australia    271.10       2           135.550000
1 2009-12-01          Austria   1998.34       2           999.170000
2 2009-12-01          Belgium    447.60       3           149.200000
3 2009-12-01  Channel Islands    989.18       1           989.180000
4 2009-12-01           Cyprus   3556.98       4           889.245000
5 2009-12-01          Denmark   1437.66       2           718.830000
6 2009-12-01             EIRE  20099.74      20          1004.987000
7 2009-12-01          Finland    549.08       1           549.080000
8 2009-12-01           France   6521.69      13           501.668462
9 2009-12-01          Germany  10096.85      18           560.936111


In [7]:
# Save the monthly analysis results as a CSV file
monthly_summary.to_csv(
    "Day5_Monthly_Revenue_by_Country.csv",
    index=False
)

print("Analysis results saved successfully!")

Analysis results saved successfully!


In [8]:
from google.colab import files

files.download("Day5_Monthly_Revenue_by_Country.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>